# Finance Dashboard — Data Exploration & Validation

**Project:** MarketCast Finance Dashboard  
**Author:** Siham Chowdhury  
**Purpose:** Explore the Azure PostgreSQL data warehouse, understand table structures, and validate key metrics against the Excel source of truth before building the dashboard.

---

## Overview

The data warehouse lives at `mc-data-warehouse.postgres.database.azure.com`. It has four schemas relevant to this project:

| Schema | Description |
|---|---|
| `core` | Clean, analytics-ready layer built on top of NetSuite and HubSpot data. **This is what we use.** |
| `hubspot` | Raw HubSpot CRM tables synced via Fivetran |
| `netsuite_netsuite` | Raw NetSuite financial tables |
| `screendragon` | Timesheet and resource management tables |

The dashboard is built on **four tables** in the `core` schema:

| Table | What it contains |
|---|---|
| `core.rpt_project_revenue_and_costs` | Main fact table — every financial transaction line (revenue and cost) |
| `core.rpt_hubspot_line_report` | Pipeline data — HubSpot deal lines |
| `core.fact_hubspot_targets` | Quarterly revenue targets by person and team |
| `core.fact_sd_timesheet_cost` | People costs — daily timesheet entries per staff member per project |

---

## 1. Setup

Connect to the database. The `get_engine()` function reads credentials from `.streamlit/secrets.toml` when running locally via Streamlit, or from `.env` when running plain Python scripts. Either way it connects to the same Azure DB.

> **Note:** Always call `engine.dispose()` after any failed query. A broken transaction leaves the connection in a `PendingRollbackError` state — disposing resets the connection pool.

In [59]:
import pandas as pd
from src.db.connection import get_engine

engine = get_engine()
engine.dispose()  # always start fresh

print("Connected to DB")

Connected to DB


---
## 2. Schema Discovery

First thing we did was discover what tables exist in the DB and which schema they live in.

Key finding: all analytics-ready tables are in the `core` schema, not `public`. This is why the initial queries failed — we had to prefix all table names with `core.`.

In [60]:
# List every table across all schemas (excluding system schemas)
pd.read_sql("""
    SELECT table_schema, table_name 
    FROM information_schema.tables
    WHERE table_type = 'BASE TABLE'
      AND table_schema NOT IN ('pg_catalog', 'information_schema')
    ORDER BY table_schema, table_name
""", engine)

,table_schema,table_name
0,core,bridge_deal_line
1,core,dim_accounting_periods
2,core,dim_accounts
3,core,dim_classes
4,core,dim_consolidated_exchange_rates
...,...,...
198,screendragon,sd_budget_by_project
199,screendragon,sd_role_rate
200,screendragon,sd_timesheet
201,screendragon,sd_timesheet_stage


---
## 3. Main Fact Table — `core.rpt_project_revenue_and_costs`

This is the most important table. Every row is a **financial transaction line** — either a revenue entry or a cost entry — tied to a specific project, client, service line, vertical, and accounting period.

### Key columns

| Column | Type | Description |
|---|---|---|
| `amount_usd` | float | The monetary value in USD. **Revenue rows are negative** (standard accounting double-entry convention — credits are negative). Always negate when summing revenue. |
| `account_type_id` | varchar | `'Income'` for revenue rows, `'COGS'` for cost rows. This is the primary filter for separating revenue from cost. |
| `accounting_period_start_date` | date | First day of the accounting period (month). Use this for time-series grouping. |
| `accounting_period_is_posted` | bool | `TRUE` means the period is finalised and posted to the ledger. **Always filter to `TRUE`** to exclude draft entries. |
| `top_level_parent_customer_name` | varchar | The ultimate parent client name (e.g. Disney, not individual subsidiaries). Use this for client-level aggregation. |
| `service_line_name` | text | The service line (Ad Solutions, Brand Tracking, Custom, Content, Data Science). |
| `vertical_name` | varchar | The vertical/industry (Theatrical/Studio, Financial Services, Auto, etc.). |
| `project_number` | varchar | Unique project identifier (e.g. PRJ55238). Used for counting distinct projects. |
| `role_name_at_timesheet_date` | varchar | Staff role when timesheets are embedded in this table (e.g. for people cost classification). |

### Important: the `Internal` vertical
Some rows have `vertical_name = 'Internal'` with costs but zero revenue. These are internal recharges and are **excluded from the dashboard** to avoid skewing profitability numbers. Needs confirmation from John Benak.

In [78]:
# Full column schema for the main fact table
pd.read_sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'core' AND table_name = 'dim_customers'
    ORDER BY ordinal_position
""", engine)

,column_name,data_type
0,customer_id,bigint
1,entity_id,character varying
2,customer_external_id,character varying
3,customer_name,character varying
4,customer_full_name,character varying
5,is_person,boolean
6,first_name,character varying
7,last_name,character varying
8,email_address,character varying
9,phone_number,character varying


In [84]:
# Sample 10 rows — notice revenue rows have negative amount_usd
# and account_type_id shows the row type (Income vs COGS)
pd.read_sql("""
    SELECT DISTINCT account_type_id, COUNT(*) as n
    FROM core.rpt_project_revenue_and_costs
    GROUP BY 1
    ORDER BY 2 DESC
""", engine)

,account_type_id,n
0,NaN,1479905
1,COGS,292378
2,Income,55772


In [61]:
# Full column schema for the main fact table
pd.read_sql("""
    SELECT column_name, data_type 
    FROM information_schema.columns 
    WHERE table_schema = 'core' AND table_name = 'rpt_project_revenue_and_costs'
    ORDER BY ordinal_position
""", engine)

,column_name,data_type
0,transaction_id,bigint
1,transaction_number,character varying
2,transaction_type,character varying
3,transaction_line_unique_key,bigint
4,transaction_document_number,character varying
5,start_date,timestamp with time zone
6,end_date,timestamp with time zone
7,project_id,bigint
8,project_number,character varying
9,project_name,character varying


In [81]:
# Sample 10 rows — notice revenue rows have negative amount_usd
# and account_type_id shows the row type (Income vs COGS)
pd.read_sql("SELECT * FROM core.rpt_project_revenue_and_costs LIMIT 10", engine)

,transaction_id,transaction_number,transaction_type,transaction_line_unique_key,transaction_document_number,start_date,end_date,project_id,project_number,project_name,...,amount,amount_usd,hours,timesheet_full_name,role_name_at_timesheet_date,is_part_of_income_statement,source_query,originating_so_line_unique_key,methodology_id,methodology_name
0,1074380,JOURNAL18729,Journal,5370598,Journal #JE18516,None,None,141788,PRJ55238,WBD Max UK-DE-IT Positioning,...,3360.00,4231.248000,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
1,1074374,JOURNAL18728,Journal,5370548,Journal #JE18515,None,None,141711,PRJ55177,Mufasa: The Lion King Exits #1 UK ES IT JP KR ...,...,37.25,46.908925,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
2,1075665,JOURNAL18755,Journal,5375154,Journal #JE18542,None,None,141992,PRJ55414,Captain America: Brave New World UK ES IT MX B...,...,60.00,75.558000,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
3,1074380,JOURNAL18729,Journal,5370604,Journal #JE18516,None,None,133370,PRJ53079,Audible BEAT Tracker 23-26,...,137.76,173.481168,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
4,1074380,JOURNAL18729,Journal,5370602,Journal #JE18516,None,None,141643,PRJ55124,Ford 2025 Global BCX Program,...,210.00,264.453000,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
5,1074381,JOURNAL18730,Journal,5370642,Journal #JE18517,None,None,141769,PRJ55224,Universal Pictures - WICKED global post-releas...,...,170.00,214.081000,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
6,1074346,JOURNAL18727,Journal,5370320,Journal #JE18514,None,None,141724,PRJ55184,LEGO P11 Pricing (US/DE) 2024,...,5117.87,6444.933691,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
7,1075665,JOURNAL18755,Journal,5375170,Journal #JE18542,None,None,141910,PRJ55337,NBA - EU Research - Qual Only,...,80.00,100.744000,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
8,1075665,JOURNAL18755,Journal,5375156,Journal #JE18542,None,None,142070,PRJ55477,Drop Short Form Test #1 DE (28205),...,240.00,302.232000,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
9,1074380,JOURNAL18729,Journal,5370607,Journal #JE18516,None,None,141939,PRJ55365,The Bad Guys 2 Ad Test Positioning # 2 MX (28093),...,68.88,86.740584,None,None,None,True,qry_vendor_bills_and_credits,None,None,None


---
## 4. Pipeline Table — `core.rpt_hubspot_line_report`

This table contains **HubSpot CRM deal lines** — each row is a deal or deal line with its current pipeline stage, value, owner, and dimensions.

### Key columns

| Column | Type | Description |
|---|---|---|
| `deal_id` | bigint | Unique identifier for the deal. One deal can have multiple rows (one per line item). |
| `deal_pipeline_stage_name` | varchar | Current stage in the pipeline (Stage 1 through Stage 5, Closed Won, Closed Lost). Pipeline queries exclude Closed Won and Closed Lost. |
| `deal_amount_usd` | float | Total deal value in USD. **This is the correct value column** — `coalesced_amount_usd` and `line_amount_usd` were found to be null for most active pipeline deals. |
| `service_line` | varchar | Service line for the deal. **Watch out:** this column sometimes contains semicolon-separated values like `"Brand Tracking;Campaign Analytics"` when a deal spans multiple service lines. Requires an explode operation in Python. |
| `is_deal_deleted` | bool | Always filter to `FALSE` to exclude soft-deleted deals. |
| `owner_full_name` | text | The deal owner's full name. |
| `vertical` | varchar | The vertical for the deal. |

### The semicolon problem
HubSpot allows tagging a deal with multiple service lines. When this happens, the `service_line` column contains a semicolon-separated string. The pipeline query pulls individual deal lines (not pre-grouped), then Python uses `str.split(";")` followed by `explode()` to turn each service line into its own row. This is handled in `src/models/financials.py` in `get_pipeline()`.

In [63]:
# Sample the pipeline table
pd.read_sql("SELECT * FROM core.rpt_hubspot_line_report LIMIT 10", engine)

,deal_line_id,deal_id,deal_name,deal_pipeline_id,deal_pipeline_stage_id,deal_pipeline_stage_name,project_id,netsuite_customer_name,hubspot_service_name,line_amount,...,vertical,sub_vertical_name,industry,sub_industry,service_line,product,deal_type,is_reccuring_baseline_revs,closed_lost_reason,is_addendum
0,4.054658e+10,22793062699,"Charles Schwab & Co., Inc. - 2025 Retirement S...",default,76355655,Closed Won,None,None,CR - Other,111500.0,...,FHAB,FHAB - Financial Services,None,None,Custom Research,1049,Annual Renewal,True,None,159.0
1,4.055493e+10,28243381341,Walmart AEE - Holiday #3,default,76355655,Closed Won,None,None,AS - AE - AE Express,36000.0,...,FHAB,FHAB - Brands,None,None,Campaign Analytics,1089,NaN,False,None,158.0
2,NaN,16415417347,NBA - Responsible Gaming Test #3,default,76355655,Closed Won,None,None,NaN,NaN,...,167,MSE - Sports & Live Events,None,None,Campaign Analytics,787,existingbusiness,False,None,NaN
3,4.047371e+10,19244354161,Paramount Pictures - A Quiet Place: Day One Tr...,default,76355655,Closed Won,None,None,AS - RTA - Trailer Report,12000.0,...,167,MSE - Sudio-Legacy,None,None,Campaign Analytics,853,existingbusiness,True,None,158.0
4,4.042948e+10,18712722196,Sony - Q2 5-2024 RTA ad hoc,default,76355655,Closed Won,None,None,AS - RTA - Trailer Report,30000.0,...,167,MSE – Studio-Growth,None,None,Campaign Analytics,853,existingbusiness,True,None,158.0
5,NaN,7907488362,3Q22 Cerulli Partnership,default,76355655,Closed Won,None,None,NaN,NaN,...,FHAB,FHAB - Financial Services,None,None,Custom Research,Synd Wealth & Affluent Monitor (WAM),Ad Hoc Renewal,None,None,159.0
6,4.055709e+10,21799607814,Paramount Smile 2 Trailer 2 24 Hour Reporting,default,76355655,Closed Won,None,None,AS - RTA - Trailer Report,12000.0,...,167,MSE - Sudio-Legacy,None,None,Campaign Analytics,853,existingbusiness,True,None,158.0
7,NaN,16328223662,Addendum - Instagram - Leadership Offsite - Re...,default,76355655,Closed Won,None,None,NaN,NaN,...,171,TTP - Platform,None,None,Custom Research,833,NaN,False,None,158.0
8,4.055426e+10,18367643746,Uber_Brand Architecture Research_Addendum 3,default,76355655,Closed Won,None,None,CR - Product Innovation & Development,1000.0,...,171,TTP - Platform,None,None,Custom Research,815,existingbusiness,False,None,158.0
9,NaN,9429367113,Jon Stewart Topic Testing for Apple TV+,default,76355655,Closed Won,None,None,NaN,NaN,...,167,MSE – Gaming/Streaming,None,None,Content,MC - CR - Content Testing - Early Concept,NaN,None,None,NaN


In [64]:
# ── Demonstrate the explode cases ─────────────────────────────

df_500 = pd.read_sql("SELECT * FROM core.rpt_hubspot_line_report LIMIT 500", engine)

# Case 1: rows where service_line contains a semicolon
# These are deals tagged with multiple service lines in HubSpot
# After explode, each service line becomes its own row
case1 = df_500[df_500["service_line"].str.contains(";", na=False)]
print(f"=== Case 1: Multi service line deals ({len(case1)} rows) ===")
print("These rows will be SPLIT by explode — one row per service line")
print(case1[["deal_id", "deal_pipeline_stage_name", "service_line", "deal_amount_usd"]].head(5))

# Case 2: rows where service_line has NO semicolon
# These are normal single-service-line deals — explode has no effect
case2 = df_500[
    ~df_500["service_line"].str.contains(";", na=False) &
    df_500["service_line"].notna()
]
print(f"\n=== Case 2: Single service line deals ({len(case2)} rows) ===")
print("These rows are UNCHANGED by explode")
print(case2[["deal_id", "deal_pipeline_stage_name", "service_line", "deal_amount_usd"]].head(5))

# Now show what happens after explode on Case 1
print("\n=== After explode on Case 1 ===")
if len(case1) > 0:
    example = case1.head(3).copy()
    print(f"Before: {len(example)} rows")
    print(example[["deal_id", "service_line", "deal_amount_usd"]])

    example["service_line"] = example["service_line"].str.split(";")
    example = example.explode("service_line")
    example["service_line"] = example["service_line"].str.strip()
    print(f"\nAfter: {len(example)} rows — same deal_id, split across service lines")
    print(example[["deal_id", "service_line", "deal_amount_usd"]])
else:
    print("No semicolon cases in this 500-row sample — try a larger sample")

=== Case 1: Multi service line deals (14 rows) ===
These rows will be SPLIT by explode — one row per service line
         deal_id deal_pipeline_stage_name  \
58   15782252844               Closed Won   
122  17768368206               Closed Won   
210  17858158166               Closed Won   
215  22888938674               Closed Won   
292  29482013861               Closed Won   

                                          service_line  deal_amount_usd  
58   Campaign Analytics;Brand Tracking;Custom Research         301900.0  
122                            Content;Custom Research         113000.0  
210                            Custom Research;Content         167000.0  
215                    Campaign Analytics;Data Science          90550.0  
292                 Custom Research;Campaign Analytics          38500.0  

=== Case 2: Single service line deals (480 rows) ===
These rows are UNCHANGED by explode
       deal_id deal_pipeline_stage_name        service_line  deal_amount_usd
0  2

---
## 5. Targets Table — `core.fact_hubspot_targets`

Quarterly revenue targets by person and team. Simple table.

### Key limitation
This table only contains data from **Q1 2025 onwards**. There is no 2024 target history in the DB. The 2026 budget lives in the Excel file (`Budget` and `Budget SL` sheets) and has not been loaded into the DB. This means Actual vs Budget cannot be built without resolving this gap with Paul/Erik.

In [65]:
# Sample targets table — note earliest date is Q1 2025
pd.read_sql("SELECT * FROM core.fact_hubspot_targets LIMIT 10", engine)

,_line,_fivetran_synced,team_primary_name,user_first_name,user_last_name,target_amount_usd,quarter_start_date,quarter_end_date
0,3,2025-06-23 21:08:19.052000+00:00,FSA,Mark,Willard,5300000,2025-10-01,2025-12-31
1,2,2025-06-23 21:08:19.052000+00:00,FSA,Mike,Chung,1100000,2025-07-01,2025-09-30
2,39,2025-06-23 21:08:19.056000+00:00,Brands - CPG,Duncan,Stark,1300000,2025-07-01,2025-09-30
3,17,2025-06-23 21:08:19.053000+00:00,Brands - CPG,Juliann,Pavlovcic,2700000,2025-01-01,2025-03-31
4,69,2025-06-23 21:08:19.058000+00:00,Europe,Christian,Dubreuil,1800000,2025-01-01,2025-03-31
5,47,2025-06-23 21:08:19.056000+00:00,FSA,Mark,Cosby,800000,2025-04-01,2025-06-30
6,75,2025-06-23 21:08:19.059000+00:00,MSE,Julanne,Schiffer,3600000,2025-04-01,2025-06-30
7,41,2025-06-23 21:08:19.056000+00:00,FSA,Mark,Sutin,0,2025-07-01,2025-09-30
8,71,2025-06-23 21:08:19.058000+00:00,MSE,David,Halas,800000,2025-07-01,2025-09-30
9,67,2025-06-23 21:08:19.058000+00:00,MSE,Dounia,Turrill,800000,2025-10-01,2025-12-31


---
## 6. People Costs Table — `core.fact_sd_timesheet_cost`

Daily timesheet entries from Screendragon — the resource management tool. Each row is one person's hours on one project on one day.

### Key columns

| Column | Type | Description |
|---|---|---|
| `netsuite_project_id` | bigint | Joins to `project_id` in `rpt_project_revenue_and_costs`. |
| `accounting_period_id` | bigint | Joins to `accounting_period_id` in the main table. |
| `timesheet_external_cost` | float | Cost at the external (billable) rate — **this is the people cost figure for profitability**. |
| `timesheet_internal_cost` | float | Cost at the internal rate — lower than external, used for internal reporting. |
| `actual_hours` | float | Hours logged for that day. Zero-hour rows exist (staff on a project but no hours logged that day). |
| `role_name_at_timesheet_date` | varchar | The person's role at the time of the timesheet (e.g. `Ops - Field Management (MK)`, `VP Research`). Used to classify labor type (Ops vs Research). |

### The join inflation problem
Joining `fact_sd_timesheet_cost` directly to `rpt_project_revenue_and_costs` inflates people costs massively because `rpt_project_revenue_and_costs` has **many transaction lines per project per period** (one per vendor bill, one per revenue recognition entry etc.). A naive join multiplies each timesheet cost row by the number of transaction lines for that project/period combination.

**The fix:** use a CTE to pre-aggregate timesheet costs to `(project, period)` grain **before** joining. See `profitability.sql` for the implementation.

### Current status
Even with the CTE fix, people costs ($68M for 2025) appear higher than COS ($31M), which is unusual for a research/services company. Needs validation with John Benak — the correct total and the correct table/columns to use need to be confirmed.

In [66]:
# Full column schema
pd.read_sql("""
    SELECT column_name, data_type 
    FROM information_schema.columns 
    WHERE table_schema = 'core' AND table_name = 'fact_sd_timesheet_cost'
    ORDER BY ordinal_position
""", engine)

,column_name,data_type
0,netsuite_project_id,bigint
1,sd_project_id,character varying
2,netsuite_project_number,character varying
3,user_id,integer
4,category,character varying
5,actual_hours,double precision
6,timesheet_date,timestamp without time zone
7,user_first_name,character varying
8,user_last_name,character varying
9,role_name_at_timesheet_date,character varying


In [67]:
# Sample — note rows 0-5 have 0.0 actual_hours but still have an external rate
# These are zero-cost days (staff on a project but not logging hours)
pd.read_sql("SELECT * FROM core.fact_sd_timesheet_cost LIMIT 10", engine)

,netsuite_project_id,sd_project_id,netsuite_project_number,user_id,category,actual_hours,timesheet_date,user_first_name,user_last_name,role_name_at_timesheet_date,...,external_rate_at_timesheet_date,timesheet_external_cost,internal_rate_at_timesheet_date,timesheet_internal_cost,accounting_period_id,accounting_period_start_date,accounting_period_ending_date,accounting_period_closed_at,accounting_period_is_posted,accounting_period_is_closed
0,133238,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-25,A,Jatin,Ops - Field Management (MK),...,137,0.0,47,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
1,133238,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-26,A,Jatin,Ops - Field Management (MK),...,137,0.0,47,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
2,133238,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-27,A,Jatin,Ops - Field Management (MK),...,137,0.0,47,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
3,133238,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-28,A,Jatin,Ops - Field Management (MK),...,137,0.0,47,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
4,133238,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-29,A,Jatin,Ops - Field Management (MK),...,137,0.0,47,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
5,133238,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-30,A,Jatin,Ops - Field Management (MK),...,137,0.0,47,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
6,133238,PRJ53001,PRJ53001,1061,Billable - Operations,4.4,2024-12-01,A,Jatin,Ops - Field Management (MK),...,137,602.8,47,206.8,136,2024-12-01,2024-12-31,2026-01-28 10:23:53+00:00,True,True
7,134645,PRJ53516,PRJ53516,1061,Billable - Operations,1.1,2024-12-02,A,Jatin,Ops - Field Management (MK),...,137,150.7,47,51.7,136,2024-12-01,2024-12-31,2026-01-28 10:23:53+00:00,True,True
8,134645,PRJ53516,PRJ53516,1061,Billable - Operations,0.0,2024-12-03,A,Jatin,Ops - Field Management (MK),...,137,0.0,47,0.0,136,2024-12-01,2024-12-31,2026-01-28 10:23:53+00:00,True,True
9,134645,PRJ53516,PRJ53516,1061,Billable - Operations,2.2,2024-12-04,A,Jatin,Ops - Field Management (MK),...,137,301.4,47,103.4,136,2024-12-01,2024-12-31,2026-01-28 10:23:53+00:00,True,True


---
## 7. Validation Against the Excel Source of Truth

The Excel file (`Revenue Data Warehouse - back up v2-2.xlsx`) is the finance team's existing reporting workbook. It is built on top of the same Azure DB using Power Pivot (Windows only). The key validation checks compare SQL query results against Excel pivot table outputs.

### Revenue logic
Revenue in the DB is stored as **negative** (standard double-entry accounting — revenue credits are negative). The DAX formula in Power Pivot confirms:
```
Revenue = -CALCULATE(SUM(rpt_project_revenue_and_costs[amount]))
```
Our SQL equivalent:
```sql
-SUM(amount_usd) WHERE account_type_id = 'Income' AND accounting_period_is_posted = TRUE
```

In [68]:
# Total 2025 revenue from the DB
# Expected: ~$135.7M based on Excel
df = pd.read_sql("""
    SELECT -SUM(amount_usd) AS total_revenue_2025
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id = 'Income'
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
""", engine)
print(f"SQL 2025 Revenue: ${df['total_revenue_2025'][0]/1e6:.1f}M")

SQL 2025 Revenue: $135.7M


In [69]:
# Total 2025 revenue from the Excel Flash - SL sheet
# Row 17 = Grand Total, columns 4-15 = months Jan-Dec 2025
# Excel values are in $000s so multiply by 1000 to compare
xl = pd.read_excel('../Revenue Data Warehouse - back up v2-2.xlsx', 
                   sheet_name='Flash - SL', header=None)

excel_2025_total = xl.iloc[17].iloc[4:16].sum()
print(f"Excel 2025 Revenue: ${excel_2025_total/1000:.1f}M (in $000s: ${excel_2025_total:,.0f}k)")

Excel 2025 Revenue: $135.7M (in $000s: $135,696k)


### Result: Revenue — ✅ VERIFIED
SQL: **$135.7M** | Excel: **$135.7M** — exact match to the penny.

This confirms:
- The `account_type_id = 'Income'` filter is correct
- Negating `amount_usd` is correct
- `accounting_period_is_posted = TRUE` is the right filter
- Our DB connection is hitting the same source as the Excel Power Pivot model

In [70]:
# Revenue by service line — SQL vs Excel comparison
# Excel values are in $000s; SQL values are in dollars
# Ad Solutions $83.1M SQL = 83,119k / 1000 = $83.1M Excel ✅

sql_sl = pd.read_sql("""
    SELECT 
        COALESCE(service_line_name, '(blank)') as service_line,
        -SUM(amount_usd) AS revenue_2025
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id = 'Income'
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1
    ORDER BY 2 DESC
""", engine)

excel_sl = xl.iloc[5:17].copy().reset_index(drop=True)
excel_sl['service_line'] = excel_sl[2].ffill()
excel_sl['revenue_2025_k'] = excel_sl.iloc[:, 4:16].sum(axis=1)
excel_sl = excel_sl[['service_line', 'revenue_2025_k']].dropna()

print("=== SQL (in $) ===")
print(sql_sl.to_string(index=False))
print("\n=== Excel (in $000s) ===")
print(excel_sl.to_string(index=False))

=== SQL (in $) ===
  service_line  revenue_2025
  Ad Solutions   83118879.11
        Custom   24618215.50
Brand Tracking   23498047.49
       Content    3133214.10
  Data Science    2161222.08
       (blank)    -834067.93

=== Excel (in $000s) ===
  service_line revenue_2025_k
  Ad Solutions    17687.06112
  Ad Solutions     3225.53275
  Ad Solutions     2852.03826
  Ad Solutions    22873.19865
  Ad Solutions    30604.37283
  Ad Solutions      5876.6755
Brand Tracking    21103.45883
Brand Tracking     2394.58866
        Custom     24618.2155
       Content      3133.2141
  Data Science     2161.22208
       (blank)     -834.06793


### Result: Revenue by Service Line — ✅ VERIFIED
After adjusting for the $000s unit difference in Excel:

| Service Line | SQL ($M) | Excel ($M) | Match |
|---|---|---|---|
| Ad Solutions | $83.1M | $83.1M | ✅ |
| Custom | $24.6M | $24.6M | ✅ |
| Brand Tracking | $23.5M | $23.5M | ✅ |
| Content | $3.1M | $3.1M | ✅ |
| Data Science | $2.2M | $2.2M | ✅ |

In [71]:
# Top 10 clients by 2025 revenue
sql_clients = pd.read_sql("""
    SELECT 
        top_level_parent_customer_name as client,
        -SUM(amount_usd) AS revenue_2025
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id = 'Income'
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    AND top_level_parent_customer_name IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""", engine)

print(sql_clients.to_string(index=False))

          client  revenue_2025
    NBCUniversal   16297211.87
          Disney   12481249.04
            Meta    9400636.75
 Ford Motor Corp    7216523.75
            LEGO    6533258.74
          Amazon    5311251.11
          Google    4136333.29
Paramount Global    3931904.25
         Walmart    3393090.58
           Chase    3387885.54


In [72]:
# Spot check specific clients against the Excel Top Clients board deck sheet
# Excel shows Apple+ $2,019k, Fox $1,915k, Fidelity $1,482k etc.
# Multiply by 1000 to compare against SQL

sql_check = pd.read_sql("""
    SELECT 
        top_level_parent_customer_name as client,
        -SUM(amount_usd) AS revenue_2025
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id = 'Income'
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    AND top_level_parent_customer_name IN (
        'Apple+', 'Fox', 'Fidelity Investments', 
        'Home Depot', 'NBCUniversal', 'Disney'
    )
    GROUP BY 1
    ORDER BY 2 DESC
""", engine)

print(sql_check.to_string(index=False))

              client  revenue_2025
        NBCUniversal   16297211.87
              Disney   12481249.04
              Apple+    2019073.40
                 Fox    1914744.07
Fidelity Investments    1482406.66
          Home Depot    1154311.54


### Result: Top Clients — ✅ VERIFIED
Client values match the Excel exactly:
- Apple+: SQL $2,019,073 = Excel $2,019k ✅
- Fox: SQL $1,914,744 = Excel $1,915k ✅
- Fidelity: SQL $1,482,407 = Excel $1,482k ✅
- Home Depot: SQL $1,154,312 = Excel $1,154k ✅

The Excel `Top clients board deck` sheet was filtered to a subset — NBCUniversal and Disney don't appear there because they're on a different filtered view, but their values also match when cross-checked.

---
## 8. People Costs Join — The Fan-Out Problem

### What went wrong
The first attempt at joining `fact_sd_timesheet_cost` to `rpt_project_revenue_and_costs` produced **$20 billion** in people costs for 2024 — obviously wrong.

### Why it happened
`rpt_project_revenue_and_costs` has **many rows per project per period** (one per transaction line — vendor bills, revenue recognition entries, etc.). A direct join means each timesheet cost row gets multiplied by the number of transaction lines for that project/period. For a project with 50 transaction lines, the people cost gets counted 50 times.

### The fix — CTE pre-aggregation
Aggregate timesheet costs to `(project, period)` grain in a CTE **before** joining. This ensures one people cost row per project per period, matching the join key grain.

See `profitability.sql` for the full implementation using three CTEs:
1. `revenue_and_cos` — aggregates revenue and COS from the main table at project/period level
2. `people` — aggregates timesheet costs at project/period level  
3. `joined` — left joins people costs onto revenue/COS, then final SELECT groups up to service line/vertical

In [73]:
# Demonstrating the fan-out problem — naive join gives $20B people cost
# DO NOT USE this in production — for illustration only
pd.read_sql("""
    WITH people_costs AS (
        SELECT
            netsuite_project_id,
            accounting_period_id,
            SUM(timesheet_external_cost) AS people_cost_usd
        FROM core.fact_sd_timesheet_cost
        WHERE accounting_period_is_posted = TRUE
        GROUP BY 1,2
    )
    SELECT 
        -SUM(CASE WHEN r.account_type_id = 'Income' THEN r.amount_usd ELSE 0 END) AS revenue_usd,
         SUM(CASE WHEN r.account_type_id = 'COGS'   THEN r.amount_usd ELSE 0 END) AS cos_usd,
        COALESCE(SUM(p.people_cost_usd), 0) AS people_cost_usd  -- still inflated!
    FROM core.rpt_project_revenue_and_costs r
    LEFT JOIN people_costs p
        ON  r.project_id = p.netsuite_project_id
        AND r.accounting_period_id = p.accounting_period_id
    WHERE r.accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM r.accounting_period_start_date) = 2024
""", engine)

,revenue_usd,cos_usd,people_cost_usd
0,1.552284e+08,4.805479e+07,2.026131e+10


---
## 9. Pipeline — The Explode Operation

### The semicolon problem
HubSpot allows tagging a deal with multiple service lines. When this happens, the `service_line` column in `rpt_hubspot_line_report` contains a semicolon-separated string like `"Brand Tracking;Campaign Analytics;Custom Research"`.

### Two distinct cases

**Case 1 — One deal, multiple service lines (semicolons in the string)**  
A single deal tagged with multiple service lines. After exploding, each service line gets its own row with the same deal value.

**Case 2 — Multiple deals, one service line (no semicolons, `num_deals > 1`)**  
Multiple separate deals that share the same stage + vertical + service line + owner combination, grouped together by the SQL.

### The pipeline SQL approach
The current pipeline query (`pipeline.sql`) pulls individual deal lines without pre-grouping, then all grouping and counting happens in Python after the explode. This avoids the ambiguity of what to do with `num_deals` when a group has both semicolons and multiple deals.

In [74]:
from pathlib import Path

# Load raw pipeline (before explode)
# Path uses .. to go up from notebooks/ to project root
sql = (Path("../src/queries") / "pipeline.sql").read_text()
df_raw = pd.read_sql(sql, engine)

print(f"Total rows before explode: {len(df_raw)}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(5)

Total rows before explode: 1317
Columns: ['deal_id', 'deal_pipeline_stage_name', 'vertical', 'service_line', 'owner_full_name', 'pipeline_value_usd']


,deal_id,deal_pipeline_stage_name,vertical,service_line,owner_full_name,pipeline_value_usd
0,39102772881,Stage 3 - Proposed,171,Campaign Analytics,Mara Doran,78000.0
1,39102772881,Stage 3 - Proposed,171,Campaign Analytics,Mara Doran,78000.0
2,39102772881,Stage 3 - Proposed,171,Campaign Analytics,Mara Doran,78000.0
3,43827334990,Stage 4 - Negotiating / Closing,FHAB,Brand Tracking,Mark Willard,84500.0
4,43827334990,Stage 4 - Negotiating / Closing,FHAB,Brand Tracking,Mark Willard,84500.0


In [75]:
# Show the explode operation step by step

# Step 1: find rows with semicolons (multi service line deals)
multi_sl = df_raw[df_raw["service_line"].str.contains(";", na=False)]
print(f"Rows with semicolons (multi service line): {len(multi_sl)}")
print(multi_sl[["deal_id", "deal_pipeline_stage_name", "service_line", "pipeline_value_usd"]].head(5))

Rows with semicolons (multi service line): 96
         deal_id deal_pipeline_stage_name                        service_line  \
102  53946434164    Stage 2 - Solutioning     Campaign Analytics;Data Science   
103  53946434164    Stage 2 - Solutioning     Campaign Analytics;Data Science   
125  22020900437  0% Prospecting Pipeline   Campaign Analytics;Brand Tracking   
132   8873819521  0% Prospecting Pipeline  Custom Research;Campaign Analytics   
147   8873790997           0% Prospecting         Brand Tracking;Data Science   

     pipeline_value_usd  
102            200000.0  
103            200000.0  
125            500000.0  
132                 0.0  
147            250000.0  


In [76]:
# Step 2: apply the explode
df_exploded = df_raw.copy()
df_exploded["service_line"] = df_exploded["service_line"].str.split(";")
df_exploded = df_exploded.explode("service_line")
df_exploded["service_line"] = df_exploded["service_line"].str.strip()

print(f"Rows after explode: {len(df_exploded)} (was {len(df_raw)})")
print(f"Extra rows created by explode: {len(df_exploded) - len(df_raw)}")

# Step 3: group correctly — count distinct deals per group
grouped = (
    df_exploded
    .groupby(["deal_pipeline_stage_name", "vertical", "service_line", "owner_full_name"])
    .agg(
        num_deals=("deal_id", "nunique"),
        pipeline_value_usd=("pipeline_value_usd", "sum")
    )
    .reset_index()
    .sort_values("pipeline_value_usd", ascending=False)
)

print(f"\nFinal grouped rows: {len(grouped)}")
grouped.head(10)

Rows after explode: 1435 (was 1317)
Extra rows created by explode: 118

Final grouped rows: 415


,deal_pipeline_stage_name,vertical,service_line,owner_full_name,num_deals,pipeline_value_usd
404,Target Account,FHAB,Campaign Analytics,Mike Chung,8,1.538000e+07
358,Stage 5 - Contracting,FHAB,Custom Research,Christian Dubreuil,3,1.491162e+07
410,Target Account,FHAB,Custom Research,Mike Chung,7,1.488000e+07
364,Target Account,167,Campaign Analytics,Ben Carlson,2,1.300000e+07
367,Target Account,167,Content,Ben Carlson,2,1.300000e+07
402,Target Account,FHAB,Campaign Analytics,Lena Kasahara,4,1.158200e+07
395,Target Account,FHAB,Brand Tracking,Lena Kasahara,4,1.158200e+07
361,Target Account,167,Brand Tracking,Ben Carlson,1,1.150000e+07
397,Target Account,FHAB,Brand Tracking,Mike Chung,5,1.055000e+07
32,0% Prospecting Pipeline,167,Campaign Analytics,Justin Bernstein,16,8.983250e+06


---
## 10. Known Gaps & Open Questions

| Item | Status | Action needed |
|---|---|---|
| Revenue total | ✅ Verified ($135.7M) | None |
| Revenue by service line | ✅ Verified | None |
| Top clients | ✅ Verified | None |
| COS | ⚠️ Built, not cross-checked | Cross-check with John |
| People costs | ❌ Suspect ($68M vs $31M COS) | Confirm correct source with John |
| Gross margin | ❌ Dependent on people costs | Fix people costs first |
| Pipeline | ⚠️ Built, no Excel equivalent | John to confirm pipeline view |
| Targets | ❌ Only Q1 2025+ in DB | Ask if 2026 budget can be loaded |
| Actual vs Budget | ❌ Not built | Blocked on budget data |
| BE / AE Synd / RTA allocations | ❌ Not built | Need fixed cost values from Erik |
| Pendings (booked not posted) | ❌ Not in DB | Ask Paul if these should be included |
| GRR / NRR retention metrics | ❌ Not built | Lower priority, post-data model sign-off |

In [91]:
sql = (Path("../src/queries") / "1_revenue_cogs_labour_by_period.sql").read_text()
df = pd.read_sql(sql, engine)
print(f"Rows: {len(df)}")


def _drop_no_client(df: pd.DataFrame) -> pd.DataFrame:
    return df[
        df["top_level_parent_customer_name"].notna() &
        (df["revenue"] > 0) &
        (df['sub_service_line_name'] == 'Brand Effect')
    ]

df = _drop_no_client(df)

df.head(10)




Rows: 32883


,accounting_period_name,accounting_period_start_date,service_line_name,sub_service_line_name,vertical_name,top_level_parent_customer_name,revenue,cogs,labour
4349,Jan 2020,2020-01-01,Ad Solutions,Brand Effect,Media,A+E Networks,3672.0,0.00,0.0
4352,Jan 2020,2020-01-01,Ad Solutions,Brand Effect,Retail-Restaurant,Alen Del Norte S.A. De C.V.,48743.0,-0.07,0.0
4355,Jan 2020,2020-01-01,Ad Solutions,Brand Effect,FSI,Allstate,50788.0,0.00,0.0
4359,Jan 2020,2020-01-01,Ad Solutions,Brand Effect,FSI,American Family Insurance,10527.0,0.00,0.0
4362,Jan 2020,2020-01-01,Ad Solutions,Brand Effect,Retail-Restaurant,ASDA Stores LTD,18542.0,481.25,0.0
4366,Jan 2020,2020-01-01,Ad Solutions,Brand Effect,Tech Telco,AT&T,208766.0,135.00,0.0
4368,Jan 2020,2020-01-01,Ad Solutions,Brand Effect,CPG,Avocados from Mexico,5550.0,-1.00,0.0
4373,Jan 2020,2020-01-01,Ad Solutions,Brand Effect,Health,Bayer,10800.0,0.00,0.0
4381,Jan 2020,2020-01-01,Ad Solutions,Brand Effect,Retail-Restaurant,Benjamin Moore & Co.,5058.0,0.00,0.0
4387,Jan 2020,2020-01-01,Ad Solutions,Brand Effect,Retail-Restaurant,Bloomin Brands,4250.0,0.00,0.0


In [92]:
# Verify Brand Effect rows — group by period and client, check counts
# Expecting count = 1 per group (one row per period/client combo)
# If count > 1 something is wrong with the grouping in the SQL

df_check = (
    df
    .groupby(["accounting_period_name", "top_level_parent_customer_name"])
    .agg(
        row_count=("revenue", "count"),
        total_revenue=("revenue", "sum")
    )
    .reset_index()
    .sort_values("row_count", ascending=False)
)

print(f"Max count per group: {df_check['row_count'].max()}")
print(f"Any count > 1: {(df_check['row_count'] > 1).any()}")
print(df_check.head(10))

Max count per group: 2
Any count > 1: True
     accounting_period_name top_level_parent_customer_name  row_count  \
3569               Oct 2022                         Disney          2   
3299               Nov 2023                   NBCUniversal          2   
3162               Nov 2021                GlaxoSmithKline          2   
453                Aug 2022                           AT&T          2   
2871               May 2022                        DirecTV          2   
1851               Jul 2022                           AT&T          2   
4012               Sep 2024                         Disney          2   
3643               Oct 2023                   NBCUniversal          2   
2286               Jun 2024                         Disney          2   
233                Apr 2024                         Disney          2   

      total_revenue  
3569       83753.34  
3299      557656.50  
3162       55729.17  
453       289454.17  
2871      160381.25  
1851      280356.47  

In [93]:
# Inspect the Disney Oct 2022 duplicate — find out why count = 2
# i.e. what's different between the two rows

df[
    (df["accounting_period_name"] == "Oct 2022") &
    (df["top_level_parent_customer_name"] == "Disney")
]

,accounting_period_name,accounting_period_start_date,service_line_name,sub_service_line_name,vertical_name,top_level_parent_customer_name,revenue,cogs,labour
13636,Oct 2022,2022-10-01,Ad Solutions,Brand Effect,Theatrical/Studio,Disney,14166.67,0.0,0.0
13637,Oct 2022,2022-10-01,Ad Solutions,Brand Effect,TV,Disney,69586.67,0.0,0.0


In [97]:
sql = (Path("../src/queries") / "2_be_allocation.sql").read_text()
df_be = pd.read_sql(sql, engine)
print(f"Rows: {len(df_be)}")

# Sanity check — sum of monthly allocations per year should equal 1000
print("\nAllocation sum per year (should equal 1000):")
print(df_be.groupby("yr")["be_allocation_per_month"].sum().round(2))

df_be.head(10)

Rows: 6372

Allocation sum per year (should equal 1000):
yr
2022    1000.20
2023     999.84
2024    1000.44
2025    1000.44
2026     999.96
Name: be_allocation_per_month, dtype: float64


,yr,accounting_period_name,accounting_period_start_date,top_level_parent_customer_name,vertical_name,annual_be_rev,annual_tot_be_rev,fixed_cost_be,be_allocation_per_month
0,2022,Jan 2022,2022-01-01,NBCUniversal,Media,2697731.75,30996067.2,1000,7.25
1,2022,Jan 2022,2022-01-01,AT&T,Tech Telco,1997963.61,30996067.2,1000,5.37
2,2022,Jan 2022,2022-01-01,AT&T,Telco & Cable,1785769.89,30996067.2,1000,4.80
3,2022,Jan 2022,2022-01-01,Fox,Media,1295640.52,30996067.2,1000,3.48
4,2022,Jan 2022,2022-01-01,Procter & Gamble,CPG,1203659.78,30996067.2,1000,3.24
5,2022,Jan 2022,2022-01-01,Walmart,Retail-Restaurant,1124126.34,30996067.2,1000,3.02
6,2022,Jan 2022,2022-01-01,NBCUniversal,Health,1013620.92,30996067.2,1000,2.73
7,2022,Jan 2022,2022-01-01,NBCUniversal,TV,972042.77,30996067.2,1000,2.61
8,2022,Jan 2022,2022-01-01,Disney,Media,681444.89,30996067.2,1000,1.83
9,2022,Jan 2022,2022-01-01,Intuit,FSI,619535.09,30996067.2,1000,1.67


In [99]:
sql = (Path("../src/queries") / "2_be_allocation.sql").read_text()
df_be = pd.read_sql(sql, engine)
print(f"Rows: {len(df_be)}")

print("\nAllocation sum per year (should equal 1000):")
print(df_be.groupby("yr")["be_allocation_per_month"].sum().round(2))

df_be.head(10)

Rows: 6372

Allocation sum per year (should equal 1000):
yr
2022    1000.20
2023     999.84
2024    1000.44
2025    1000.44
2026     999.96
Name: be_allocation_per_month, dtype: float64


,yr,accounting_period_name,accounting_period_start_date,top_level_parent_customer_name,vertical_name,service_line_name,sub_service_line_name,annual_be_rev,annual_tot_be_rev,fixed_cost_be,be_allocation_per_month
0,2022,Jan 2022,2022-01-01,NBCUniversal,Media,Ad Solutions,Brand Effect,2697731.75,30996067.2,1000,7.25
1,2022,Jan 2022,2022-01-01,AT&T,Tech Telco,Ad Solutions,Brand Effect,1997963.61,30996067.2,1000,5.37
2,2022,Jan 2022,2022-01-01,AT&T,Telco & Cable,Ad Solutions,Brand Effect,1785769.89,30996067.2,1000,4.80
3,2022,Jan 2022,2022-01-01,Fox,Media,Ad Solutions,Brand Effect,1295640.52,30996067.2,1000,3.48
4,2022,Jan 2022,2022-01-01,Procter & Gamble,CPG,Ad Solutions,Brand Effect,1203659.78,30996067.2,1000,3.24
5,2022,Jan 2022,2022-01-01,Walmart,Retail-Restaurant,Ad Solutions,Brand Effect,1124126.34,30996067.2,1000,3.02
6,2022,Jan 2022,2022-01-01,NBCUniversal,Health,Ad Solutions,Brand Effect,1013620.92,30996067.2,1000,2.73
7,2022,Jan 2022,2022-01-01,NBCUniversal,TV,Ad Solutions,Brand Effect,972042.77,30996067.2,1000,2.61
8,2022,Jan 2022,2022-01-01,Disney,Media,Ad Solutions,Brand Effect,681444.89,30996067.2,1000,1.83
9,2022,Jan 2022,2022-01-01,Intuit,FSI,Ad Solutions,Brand Effect,619535.09,30996067.2,1000,1.67


In [106]:
# Disney BE allocation example — walk through the maths row by row

# Step 1: Disney's annual BE revenue per year
disney_annual = df_be[
    df_be["top_level_parent_customer_name"] == "Disney"
][["yr", "vertical_name", "annual_be_rev", "annual_tot_be_rev",
   "fixed_cost_be", "be_allocation_per_month"]].drop_duplicates(
    subset=["yr", "vertical_name"]
).sort_values(["yr", "vertical_name"])

print("=== Disney Annual BE Revenue & Allocation ===")
print(disney_annual.to_string(index=False))

# Step 2: show the maths explicitly for one year
yr = 2025
row = disney_annual[disney_annual["yr"] == yr].iloc[0]
print(f"\n=== Manual check for Disney {yr} ===")
print(f"Disney annual BE rev:    ${row['annual_be_rev']:,.2f}")
print(f"Total BE rev all clients:${row['annual_tot_be_rev']:,.2f}")
print(f"Disney share:            {row['annual_be_rev']/row['annual_tot_be_rev']*100:.2f}%")
print(f"Fixed cost:              ${row['fixed_cost_be']:,.0f}")
print(f"Annual allocation:       ${row['annual_be_rev']/row['annual_tot_be_rev']*row['fixed_cost_be']:.2f}")
print(f"Monthly allocation (/12):${row['be_allocation_per_month']:.2f}")

# Step 3: confirm Disney appears in all 12 months for that year
disney_months = df_be[
    (df_be["top_level_parent_customer_name"] == "Disney") &
    (df_be["yr"] == yr)
][["accounting_period_name", "be_allocation_per_month"]]
print(f"\n=== Disney {yr} — all {len(disney_months)} months ===")
print(disney_months.to_string(index=False))
print(f"Sum of monthly allocations: ${disney_months['be_allocation_per_month'].sum():.2f}")

=== Disney Annual BE Revenue & Allocation ===
  yr     vertical_name  annual_be_rev  annual_tot_be_rev  fixed_cost_be  be_allocation_per_month
2022             Media      695257.39        30996067.20           1000                     1.87
2022                TV      288446.34        30996067.20           1000                     0.78
2022 Theatrical/Studio       35833.33        30996067.20           1000                     0.10
2022               NaN      -39986.56        30996067.20           1000                    -0.11
2023         Streaming       40000.00        28962819.57           1000                     0.12
2023                TV      974450.00        28962819.57           1000                     2.80
2024         Streaming      120000.00        26026658.50           1000                     0.38
2024                TV      988500.00        26026658.50           1000                     3.17
2025         Streaming       40000.00        22873198.65           1000          

In [107]:
# Check Disney 2025 sum by vertical — each should equal annual allocation
disney_2025 = df_be[
    (df_be["top_level_parent_customer_name"] == "Disney") &
    (df_be["yr"] == 2025)
]

print("Disney 2025 — sum by vertical (should match annual allocation * 12 / 12 = monthly * 12):")
print(
    disney_2025
    .groupby("vertical_name")
    .agg(
        monthly_rate=("be_allocation_per_month", "first"),
        total_allocated=("be_allocation_per_month", "sum"),
        months=("be_allocation_per_month", "count")
    )
)

print(f"\nGrand total allocated to Disney 2025: ${disney_2025['be_allocation_per_month'].sum():.2f}")
print(f"Disney's share of total annual BE cost: ${disney_2025['be_allocation_per_month'].sum() * 12 / 12:.2f}")

Disney 2025 — sum by vertical (should match annual allocation * 12 / 12 = monthly * 12):
               monthly_rate  total_allocated  months
vertical_name                                       
Streaming              0.15             1.80      12
TV                     3.71            44.52      12

Grand total allocated to Disney 2025: $46.32
Disney's share of total annual BE cost: $46.32


In [108]:
# Show all distinct combinations for Disney TV 2025
df_be[
    (df_be["top_level_parent_customer_name"] == "Disney") &
    (df_be["yr"] == 2025) &
    (df_be["vertical_name"] == "TV")
][["accounting_period_name", "vertical_name", "service_line_name",
   "sub_service_line_name", "be_allocation_per_month"]].head(24)

,accounting_period_name,vertical_name,service_line_name,sub_service_line_name,be_allocation_per_month
4494,Jan 2025,TV,Ad Solutions,Brand Effect,3.71
4546,Feb 2025,TV,Ad Solutions,Brand Effect,3.71
4598,Mar 2025,TV,Ad Solutions,Brand Effect,3.71
4650,Apr 2025,TV,Ad Solutions,Brand Effect,3.71
4702,May 2025,TV,Ad Solutions,Brand Effect,3.71
4754,Jun 2025,TV,Ad Solutions,Brand Effect,3.71
4806,Jul 2025,TV,Ad Solutions,Brand Effect,3.71
4858,Aug 2025,TV,Ad Solutions,Brand Effect,3.71
4910,Sep 2025,TV,Ad Solutions,Brand Effect,3.71
4962,Oct 2025,TV,Ad Solutions,Brand Effect,3.71


In [104]:
# Check how many distinct customer_ids map to Disney
pd.read_sql("""
    SELECT
        c.top_level_parent_customer_name,
        r.customer_id,
        c.customer_full_name,
        -SUM(r.amount) AS annual_be_rev
    FROM core.rpt_project_revenue_and_costs r
    LEFT JOIN core.dim_customers c ON r.customer_id = c.customer_id
    WHERE r.account_type_id = 'Income'
      AND r.sub_service_line_name = 'Brand Effect'
      AND r.accounting_period_is_posted = TRUE
      AND c.top_level_parent_customer_name = 'Disney'
      AND EXTRACT(YEAR FROM r.accounting_period_start_date) = 2025
    GROUP BY 1, 2, 3
    ORDER BY 4 DESC
""", engine)

,top_level_parent_customer_name,customer_id,customer_full_name,annual_be_rev
0,Disney,44119,Disney : The Walt Disney Company,518962.5
1,Disney,4355,Disney,499250.0
2,Disney,4769,Disney : Hulu,40000.0


In [105]:
sql = (Path("../src/queries") / "2_be_allocation.sql").read_text()
df_be = pd.read_sql(sql, engine)

print("Allocation sum per year (should equal 1000):")
print(df_be.groupby("yr")["be_allocation_per_month"].sum().round(2))

# Disney 2025 check — should now show exactly 12 rows per vertical
disney_2025 = df_be[
    (df_be["top_level_parent_customer_name"] == "Disney") &
    (df_be["yr"] == 2025)
]
print(f"\nDisney 2025 rows: {len(disney_2025)} (expect 12 per vertical)")
print(disney_2025.groupby("vertical_name")["be_allocation_per_month"].agg(["count", "sum"]))

Allocation sum per year (should equal 1000):
yr
2022    1000.20
2023     999.96
2024    1000.32
2025    1000.44
2026     999.96
Name: be_allocation_per_month, dtype: float64

Disney 2025 rows: 24 (expect 12 per vertical)
               count    sum
vertical_name              
Streaming         12   1.80
TV                12  44.52


In [112]:
df_be[
    (df_be["top_level_parent_customer_name"] == "Disney") ]

,yr,accounting_period_name,accounting_period_start_date,top_level_parent_customer_name,vertical_name,service_line_name,sub_service_line_name,annual_be_rev,annual_tot_be_rev,fixed_cost_be,be_allocation_per_month
8,2022,Jan 2022,2022-01-01,Disney,Media,Ad Solutions,Brand Effect,695257.39,30996067.20,1000,1.87
32,2022,Jan 2022,2022-01-01,Disney,TV,Ad Solutions,Brand Effect,288446.34,30996067.20,1000,0.78
107,2022,Jan 2022,2022-01-01,Disney,Theatrical/Studio,Ad Solutions,Brand Effect,35833.33,30996067.20,1000,0.10
207,2022,Jan 2022,2022-01-01,Disney,NaN,Ad Solutions,Brand Effect,-39986.56,30996067.20,1000,-0.11
224,2022,Feb 2022,2022-02-01,Disney,Media,Ad Solutions,Brand Effect,695257.39,30996067.20,1000,1.87
...,...,...,...,...,...,...,...,...,...,...,...
5551,2026,Oct 2026,2026-10-01,Disney,Streaming,Ad Solutions,Brand Effect,50000.00,17953207.57,1000,0.23
5564,2026,Nov 2026,2026-11-01,Disney,TV,Ad Solutions,Brand Effect,1048304.50,17953207.57,1000,4.87
5596,2026,Nov 2026,2026-11-01,Disney,Streaming,Ad Solutions,Brand Effect,50000.00,17953207.57,1000,0.23
5609,2026,Dec 2026,2026-12-01,Disney,TV,Ad Solutions,Brand Effect,1048304.50,17953207.57,1000,4.87


In [159]:
sql = (Path("../src/queries") / "5_gross_margin.sql").read_text()
df_gm = pd.read_sql(sql, engine)
print(f"Rows: {len(df_gm)}")

# Sanity check vs Excel — 2025 totals
totals = df_gm[pd.to_datetime(df_gm["accounting_period_start_date"]).dt.year == 2025].agg({
    "revenue": "sum",
    "cogs": "sum",
    "be_allocation": "sum",
    "ae_allocation": "sum",
    "rta_allocation": "sum",
    "gross_margin": "sum"
}).round(0)

print("\n2025 totals:")
print(totals)
print(f"\nGM %: {totals['gross_margin']/totals['revenue']*100:.1f}%")

df_gm.head(10)

Rows: 38809

2025 totals:
revenue           135695510.0
cogs               45190818.0
be_allocation      10158000.0
ae_allocation        938000.0
rta_allocation       833000.0
gross_margin       78575692.0
dtype: float64

GM %: 57.9%


,accounting_period_name,accounting_period_start_date,top_level_parent_customer_name,vertical_name,service_line_name,sub_service_line_name,revenue,cogs,labour,be_allocation,ae_allocation,rta_allocation,gross_margin,gm_pct
0,Jan 2017,2017-01-01,Unassigned,Client Advisor,NaN,NaN,148946.00,74872.39,0.0,0.0,0.0,0.0,74073.61,49.7
1,Jan 2017,2017-01-01,Unassigned,NaN,NaN,NaN,3162361.13,977157.94,0.0,0.0,0.0,0.0,2185203.19,69.1
2,Feb 2017,2017-02-01,Unassigned,Client Advisor,NaN,NaN,88140.95,50448.21,0.0,0.0,0.0,0.0,37692.74,42.8
3,Feb 2017,2017-02-01,Unassigned,NaN,NaN,NaN,2594102.14,555997.88,0.0,0.0,0.0,0.0,2038104.26,78.6
4,Mar 2017,2017-03-01,Unassigned,Client Advisor,NaN,NaN,75917.00,51972.68,0.0,0.0,0.0,0.0,23944.32,31.5
5,Mar 2017,2017-03-01,Unassigned,NaN,NaN,NaN,2953058.68,840826.62,0.0,0.0,0.0,0.0,2112232.06,71.5
6,Apr 2017,2017-04-01,Unassigned,Client Advisor,NaN,NaN,112477.00,39966.28,0.0,0.0,0.0,0.0,72510.72,64.5
7,Apr 2017,2017-04-01,Unassigned,NaN,NaN,NaN,2850575.13,710748.33,0.0,0.0,0.0,0.0,2139826.80,75.1
8,May 2017,2017-05-01,Unassigned,Client Advisor,NaN,NaN,159658.00,59037.09,0.0,0.0,0.0,0.0,100620.91,63.0
9,May 2017,2017-05-01,Unassigned,NaN,NaN,NaN,2877048.70,790031.88,0.0,0.0,0.0,0.0,2087016.82,72.5


In [118]:
totals_2025 = df_gm[
    pd.to_datetime(df_gm["accounting_period_start_date"]).dt.year == 2025
].agg({
    "revenue": "sum",
    "cogs": "sum",
    "labour": "sum"
}).round(0)

print(totals_2025)

revenue    136035263.0
cogs        30719923.0
labour             0.0
dtype: float64


In [115]:
# Quick check — does using amount vs amount_usd make a difference?
pd.read_sql("""
    SELECT
        -SUM(amount)     AS total_amount,
        -SUM(amount_usd) AS total_amount_usd
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id = 'Income'
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
""", engine)

,total_amount,total_amount_usd
0,1.356955e+08,1.356955e+08


In [116]:
pd.read_sql("""
    SELECT
        SUM(CASE WHEN account_type_id = 'COGS' THEN amount ELSE 0 END) AS cogs_only,
        SUM(CASE WHEN account_type_id IS NULL   THEN amount ELSE 0 END) AS labour_only,
        SUM(CASE WHEN account_type_id IN ('COGS') OR account_type_id IS NULL THEN amount ELSE 0 END) AS cogs_plus_labour
    FROM core.rpt_project_revenue_and_costs
    WHERE accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
""", engine)

,cogs_only,labour_only,cogs_plus_labour
0,45190818.01,34578814.9,7.976963e+07


In [119]:
# How many rows are being dropped by the customer join?
pd.read_sql("""
    SELECT
        CASE WHEN c.top_level_parent_customer_name IS NULL
             THEN 'No customer match'
             ELSE 'Has customer' END AS customer_status,
        account_type_id,
        COUNT(*) as rows,
        SUM(amount) as total_amount
    FROM core.rpt_project_revenue_and_costs r
    LEFT JOIN core.dim_customers c ON r.customer_id = c.customer_id
    WHERE r.accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM r.accounting_period_start_date) = 2025
    GROUP BY 1, 2
    ORDER BY 1, 2
""", engine)

,customer_status,account_type_id,rows,total_amount
0,Has customer,COGS,38403,4.505872e+07
1,Has customer,Income,6264,-1.360353e+08
2,Has customer,NaN,702735,2.864932e+07
3,No customer match,COGS,1479,1.320971e+05
4,No customer match,Income,331,3.397529e+05
5,No customer match,NaN,83785,5.929490e+06


In [132]:
sql = (Path("../src/queries") / "5_gross_margin.sql").read_text()
df_gm = pd.read_sql(sql, engine)

totals_2025 = df_gm[
    pd.to_datetime(df_gm["accounting_period_start_date"]).dt.year == 2025
].agg({
    "revenue": "sum",
    "cogs": "sum",
    "labour": "sum"
}).round(0)

print(totals_2025)

revenue    135695510.0
cogs        45190818.0
labour      34578815.0
dtype: float64


In [161]:

sql = (Path("../src/queries") / "5_gross_margin.sql").read_text()
df_gm = pd.read_sql(sql, engine)

df_gm_be = df_gm[
    (pd.to_datetime(df_gm["accounting_period_start_date"]).dt.year == 2025) &
    (df_gm["sub_service_line_name"] == "Brand Effect")
].agg({
    "revenue": "sum",
    "cogs": "sum",
    "gross_margin": "sum",
    "labour": "sum",
    "be_allocation": "sum",
    "ae_allocation": "sum",
    "rta_allocation": "sum"
}).round(0)

print(df_gm_be)
print(f"GM%: {df_gm_be['gross_margin']/df_gm_be['revenue']*100:.1f}%")

revenue           135695510.0
cogs               45190818.0
gross_margin       78575692.0
labour             34578815.0
be_allocation      10158000.0
ae_allocation        938000.0
rta_allocation       833000.0
dtype: float64
GM%: 57.9%


In [153]:
# Check what the be_allocation sums to across ALL rows for 2025
# not just Brand Effect rows
df_gm_2025 = df_gm[pd.to_datetime(df_gm["accounting_period_start_date"]).dt.year == 2025]

print("be_allocation sum across ALL 2025 rows:")
print(df_gm_2025["be_allocation"].sum().round(0))

print("\nbe_allocation sum for Brand Effect rows only:")
print(df_gm_2025[df_gm_2025["sub_service_line_name"] == "Brand Effect"]["be_allocation"].sum().round(0))

print("\nbe_allocation sum for non-Brand Effect rows:")
print(df_gm_2025[df_gm_2025["sub_service_line_name"] != "Brand Effect"]["be_allocation"].sum().round(0))

be_allocation sum across ALL 2025 rows:
9807136.0

be_allocation sum for Brand Effect rows only:
9807136.0

be_allocation sum for non-Brand Effect rows:
0.0


In [158]:

sql = (Path("../src/queries") / "5_gross_margin.sql").read_text()
df_gm = pd.read_sql(sql, engine)

df_gm_be = df_gm[
    (pd.to_datetime(df_gm["accounting_period_start_date"]).dt.year == 2025) &
    (df_gm["top_level_parent_customer_name"] == "Chase")
].agg({
    "revenue": "sum",
    "cogs": "sum",
    "gross_margin": "sum",
    "labour": "sum",
    "be_allocation": "sum",
    "ae_allocation": "sum",
    "rta_allocation": "sum"

}).round(0)

print(df_gm_be)
print(f"GM%: {df_gm_be['gross_margin']/df_gm_be['revenue']*100:.1f}%")

revenue           3387886.0
cogs               608496.0
gross_margin      2599431.0
labour            1022749.0
be_allocation           0.0
ae_allocation      169956.0
rta_allocation      10002.0
dtype: float64
GM%: 76.7%


In [151]:
# What does the raw be_alloc CTE produce for 2025?
pd.read_sql("""
    WITH fixed_costs AS (
        SELECT 2025 AS yr, 10158000 AS fixed_cost_be
    ),
    be_rev AS (
        SELECT
            EXTRACT(YEAR FROM r.accounting_period_start_date)::int AS yr,
            COALESCE(c.top_level_parent_customer_name, 'Unassigned') AS top_level_parent_customer_name,
            r.vertical_name, r.service_line_name, r.sub_service_line_name,
            -SUM(r.amount) AS annual_be_rev
        FROM core.rpt_project_revenue_and_costs r
        LEFT JOIN core.dim_customers c ON r.customer_id = c.customer_id
        WHERE r.account_type_id = 'Income'
          AND r.sub_service_line_name = 'Brand Effect'
          AND r.accounting_period_is_posted = TRUE
          AND EXTRACT(YEAR FROM r.accounting_period_start_date) = 2025
        GROUP BY 1,2,3,4,5
    ),
    tot AS (SELECT yr, SUM(annual_be_rev) AS tot FROM be_rev GROUP BY 1),
    periods AS (
        SELECT DISTINCT
            EXTRACT(YEAR FROM accounting_period_start_date)::int AS yr,
            accounting_period_name
        FROM core.rpt_project_revenue_and_costs
        WHERE accounting_period_is_posted = TRUE
          AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    )
    SELECT
        SUM((b.annual_be_rev / NULLIF(t.tot,0)) * f.fixed_cost_be / 12) AS total_be_alloc
    FROM be_rev b
    JOIN tot t ON b.yr = t.yr
    JOIN fixed_costs f ON b.yr = f.yr
    CROSS JOIN periods p
""", engine)

,total_be_alloc
0,10158000.0


In [160]:
# Check dim_customers for a 'have' or similar flag column
pd.read_sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'core'
    AND table_name = 'dim_customers'
    ORDER BY ordinal_position
""", engine)

,column_name,data_type
0,customer_id,bigint
1,entity_id,character varying
2,customer_external_id,character varying
3,customer_name,character varying
4,customer_full_name,character varying
5,is_person,boolean
6,first_name,character varying
7,last_name,character varying
8,email_address,character varying
9,phone_number,character varying
